In [26]:
import snappy

class NoSolutions(Exception):
    pass

class NoManifold(Exception):
    pass

class MoreInput(Exception):
    pass

class TimeOut(Exception):
    pass

class TAP():
    def __init__(self, manifold:str = None, dimension:int = 2, lie_algebra:str = "PSL", obstruction = 1, solution = None, engine:str = "magma", twister = False, tsurface = 'S_1_1'):
        self.set_manifold(manifold, twister, tsurface)
        self.set_dimension(dimension)
        self.set_lie_algebra(lie_algebra)
        self.set_ptolemy_solutions(engine, obstruction)
#         self.set_solution(solution)
#         self.set_number_field()
#         self.set_polynomial_ring()
        self.init_variables()
        if solution:
            self.set_tap(solution)
        return None
    
    def init_variables(self):
        self.init_fundamental_group()
        self.init_generators()
        self.init_relators()
        self.init_free_algebra()
        self.init_cusp_variables()
        return None
    
    #Set functions
    def init_cusp_variables(self):
        self.__cusp = []
        preabel = self.preabelianize()
        for i in range(len(preabel)):
            self.__cusp.append('t{}'.format(i))
        
    def set_manifold(self,m: str = None, twister = False, tsurface = 'S_1_1'):
        if m and twister == False:
            self.__manifold = snappy.Manifold(m)
        elif twister:
            self.__manifold = snappy.twister.Surface(tsurface).bundle(monodromy=m)
        else:
            raise NoManifold("Please specify a manifold")
        return None
    
    def set_dimension(self, dim:int = 2):
        self.__dimension = dim
        return None
    
    def set_lie_algebra(self, lie_algebra:str):
        self.__lie_algebra = lie_algebra
        return None
    
    def set_ptolemy_solutions(self, opt:str = "magma", obstr = 1):
        param = (self.dimension(), obstr)
        if self.lie_algebra() == "SL":
            param = (self.dimension(),)
        if opt == "r":
            self.__ptolemy_solutions = self.manifold().ptolemy_variety(*param).retrieve_solutions(verbose = False)
        elif opt == "sage":
            self.__ptolemy_solutions = self.manifold().ptolemy_variety(*param).compute_solutions(engine = 'sage')
        elif opt == "magma":
            self.__ptolemy_solutions = self.manifold().ptolemy_variety(*param).compute_solutions(engine = 'magma', verbose = False)
        return None  
    
    def set_solution(self, sol:int):
        self.__solution = sol
        return None
    
    def init_fundamental_group(self):
        self.__fundamental_group = self.manifold().fundamental_group()
        return None
    
    def init_generators(self):
        self.__generators = self.fundamental_group().generators()
        return None
    
    def init_relators(self):
        self.__relators = self.fundamental_group().relators()
        return None
    
    def init_free_algebra(self):
        self.__free_algebra = self.make_alg()
        return None
    
    def init_number_field(self): 
        if not self.ptolemy_solutions():
            raise NoSolutions("There are no irreducible, parabolic representations for this manifold in {}({},C). Please try a different manifold or target space.".format(self.lie_algebra(), self.dimension()))
        self.__number_field = self.make_numfld()
        return None
    
    def init_polynomial_ring(self):
        self.__polynomial_ring = self.make_polyring()
        return None
    
    def set_tap(self, solution):
        self.__tap = self.make_tap(solution)
        return None
    
    
    #Get functions
    def cusp(self):
        return self.__cusp
    
    def manifold(self):
        return self.__manifold
    
    def dimension(self):
        return self.__dimension
    
    def lie_algebra(self):
        return self.__lie_algebra
    
    def ptolemy_solutions(self):
        return self.__ptolemy_solutions
    
    def solution(self):
        return self.__solution
    
    def fundamental_group(self):
        return self.__fundamental_group
    
    def generators(self):
        return self.__generators
    
    def relators(self):
        return self.__relators
    
    def free_algebra(self):
        return self.__free_algebra
    
    def number_field(self):
        return self.__number_field
    
    def polynomial_ring(self):
        return self.__polynomial_ring
    
    def tap(self):
        return self.__tap
    
    #Make functions

    #Generates a sage FreeAlgebra from an array of variables
    def make_alg(self):
        gens = self.generators()
        fagens = "Id"
        for i in gens:
            fagens = fagens + "," + i + "," + i.upper()
        F = FreeAlgebra(ZZ, 2*len(gens) + 1, fagens)
        F.inject_variables(verbose = False)
        return F
        
    #Generates the numberfield over which the TAP polynomial will live
    def make_numfld(self, sol:int = None):
        if sol == None:
            sol = self.solution()
        x = SR.var('x')
        pol = ""
        if self.ptolemy_solutions():
            for i in range(self.dimension()):
                for j in range(self.dimension()):
                    try:
                        pol = self.ptolemy_solutions()[sol].evaluate_word('a')[i][j].mod().Str().python()
                    except:
                        pol = ""
            if pol == "":
                numfld = IntegerRing()
            else:
                numfld.<x> = NumberField(SR(pol))
                numfld.inject_variables(verbose = False)
            return numfld
        else:
            return
    
    #Generates the polynomial ring in which the TAP lives (for now this will only work with orientable 1-cusp manifolds)
    def make_polyring(self):
        polyring = LaurentPolynomialRing(self.number_field(), self.cusp())
        polyring.inject_variables(verbose = False)
        return polyring
    
    
    ##########################################################################################################################################################
    #All non-initialization methods are below
    
    #This will return mapping information for the abelianization of the fundamental group. 
    def abelianize(self):
        polyring = self.polynomial_ring()
        cusp = self.cusp()
        solutions = self.preabelianize()
        abelianize = []
        for i in range(len(self.generators())):
            abgen = 1
            for j in range(len(solutions)):
                abgen = abgen*polyring(cusp[j])^solutions[j][i]
            abelianize.append(abgen)
        return abelianize
    
    
    #This will find the total power of each letter in a string of characters, assuming capital letters as inverses.
    def find_powers(self, polynomial):
        pol="{}".format(polynomial)
        power = 0
        powers = []
        for gen in self.generators():
            power = 0
            for factor in pol:
                if factor.lower() == gen:
                    if factor.islower():
                        power += 1
                    else:
                        power -= 1
            powers.append(power)        
        return powers
    
    
    #Computes the fox derivative of a word with respect to a given character.
    #Assumes capital letters as inverse
    def fox_der(self, wrd: str, let: str):
        out = ""
        if wrd[0] == let:
            out = out + "Id"
        elif wrd[0] == let.upper():
            out = out + "-Id" + self.make_mult(wrd[0])
        for i in range(1,len(wrd)):
            if wrd[i] == let:
                out = out + "+Id" + self.make_mult(wrd[:i])
            elif wrd[i] == let.upper():
                out = out + "-Id" + self.make_mult(wrd[:i+1])
        if out == "":
            out = "0"
        return out
    

    #Takes a snappy Manifold object and returns a sage matrix Fox Jacobian
    def fox_jacobi(self):
        gens = self.generators()
        rels = self.relators()
        F = self.free_algebra()
        M = MatrixSpace(F, len(rels), len(gens))
        out = M.matrix()
        i = 0
        for rel in rels:
            j = 0
            for gen in gens:
                out[i,j] = self.fox_der(rel, gen)
                j += 1
            i += 1
        return out
    
    
    #Converts the fox jacobian into a flattened matrix of matrices
    def fox_jacobi_to_twisted_alexander_matrix(self, gen:str = ""):
        if not self.ptolemy_solutions():
            return matrix(0)
        maps = self.lift_rep()
        gens = self.generators()
        rels = self.relators()
        dim = self.dimension()
        pol = self.polynomial_ring()
        mapsext = [1]
        genlist = ["Id"]
        for i in gens:
            genlist.append(i)
            genlist.append(i.upper())
            mapsext.append(pol(maps[gens.index(i)]))
            mapsext.append(pol(maps[gens.index(i)]^-1))
        replist = []
        for i in genlist:
            replist.append(mapsext[genlist.index(i)]*self.make_rep(gen = i))
        
        replist = tuple(replist)
        preblockmat = self.reduce_fox_jacobi(gen)
        blocklist = []
        for i in range(len(rels)):
            for j in range(len(gens)-1):
                blocklist.append(preblockmat[i,j](replist)) 
        tapmatspace = MatrixSpace(self.polynomial_ring(),len(rels)*dim,(len(gens)-1)*dim)
        twisted_alexander_matrix = tapmatspace(0)
        for i in range(len(rels)*dim):
            for j in range((len(gens)-1)*dim):
                twisted_alexander_matrix[i,j] = blocklist[(j//dim)+(i-i%dim)][i%dim,j%dim] 
        return twisted_alexander_matrix
    
    
    #This will contain lift information of the PSL representation into SL.
    def lift_rep(self):
        #First we pull the abelianization of the fundamental group.  We will be appending our sign changes to this.
        abelian = self.abelianize()
        if self.lie_algebra() == "SL":
            return abelian
        
        gens = self.generators()
        
        rels = []
        fa_rel = []
        for i in self.relators():
            rels.append("1" + self.make_mult(i)) 
            fa_rel.append(self.free_algebra()("1" + self.make_mult(i)))
                
        dim = self.dimension()
        polyring = self.polynomial_ring()
        matspc = MatrixSpace(polyring, dim)
        
        genlist = ["Id"]
        for i in gens:
            genlist.append(i)
            genlist.append(i.upper())
            
        replist = []
        for i in genlist:
            replist.append(self.make_rep(gen = i))
        
        replist = tuple(replist)
        #Here we determine whether the relator multiplies to Id or -Id in SL.  If it multiplies to the identity, we assign 0. Otherwise we assign 1.
        mults = []
        for r in fa_rel:
            if r(replist) == matspc.identity_matrix():
                mults.append(0)
            else:
                mults.append(1)
        mults = vector(Integers(2), mults)
        
        #We create a Z/2Z matrix space with rows = number of relators and generators = number of generators.
        #If a generator shows up an even number of times in a relator, we set the corresponding entry to 0, otherwise we set it to odd.
        #We want to solve a system of equations of the form x*a + y*b + z*c = 0 or x*a + y*b + z*c = 1 where x,y,z are unknowns and a,b,c are elements of Z/2Z set in the previous step.
        #The right side of each equation is determined by the vector mults.
        #Effectively we are solving how we can enact a sign change where we need one, and leave the sign untouched where we don't want a sign change.
        powerspc = MatrixSpace(Integers(2), len(self.relators()), len(self.generators()))
        powers = powerspc(0)
        for i in range(len(self.relators())):
            for j in range(len(self.generators())):
                powers[i,j] = (self.find_powers(rels[i])[j]%2)
                
        solutions = powers.solve_right(mults)
        
        for i in range(len(abelian)):
            if solutions[i] == 1:
                abelian[i] = -abelian[i]

        return abelian
    
    
    #Adds an asterisk between characters
    #For use with fox_der method
    def make_mult(self, wrd: str):
        out = ""
        for i in range(len(wrd)):
            out = out + "*" + wrd[i]
        return out
    
    
    #Creates a representation from the Ptolemy Variety as a sage matrix over a number field
    def make_rep(self, gen: str = "", solution:int = None):
        if solution != None:
            self.set_solution(solution)
            self.init_number_field()
            self.init_polynomial_ring()
        dim = self.dimension()
        ptolemysol = self.ptolemy_solutions()[self.solution()]
        if not ptolemysol:
            return matrix(dim)
        if gen == "":
            raise MoreInput("A generator or word is required to run the make_rep() method.")
        grp = self.fundamental_group()
        polyring = self.polynomial_ring()
        matspc = MatrixSpace(polyring, dim)
        if gen == "Id":
            return matspc.identity_matrix()    
        rep = matspc.matrix()
        i=0
        for i in range(dim):
            j=0
            for j in range(dim):
                try:
                    rep[i,j] = polyring(ptolemysol.evaluate_word(gen, grp)[i][j].nf_get_pol().Str().python())
                except:
                    rep[i,j] = ptolemysol.evaluate_word(gen, grp)[i][j]
                j += 1
            i += 1
        return rep
    
    
    #The ultimate purpose of this class. Computes the TAP after picking a Ptolemy variety solution.
    def make_tap(self, sol:int = 0, gen:str = ""):
        self.set_solution(sol)
        self.init_number_field()
        self.init_polynomial_ring()
        
        if not gen:
            power = self.preabelianize()[0]  #Temporary: works for knots, needs to be refined for links
            i = 0
            while i < len(power):
                if power[i] > 0:
                    gen = self.generators()[i]
                    break
                i += 1
                if i == len(power):
                    i = 0
                    for g in range(len(power)):
                        power[g] = 1/power[g]
                
        maps = self.lift_rep()
        numerator = self.fox_jacobi_to_twisted_alexander_matrix(gen).det()
        denominator = (identity_matrix(self.dimension()) - maps[self.generators().index(gen)]*self.make_rep(gen)).det()
        return (numerator/denominator).numerator()
    
    
    #Finds the power of t which each generator will be sent to.
    def preabelianize(self):
        rels = self.relators()
        matspc = MatrixSpace(IntegerRing(), len(self.relators()), len(self.generators()))
        powers = matspc(0)
        for i in range(len(self.relators())):
            for j in range(len(self.generators())):
                powers[i,j] = self.find_powers(rels[i])[j]
        solutions = powers.right_kernel().basis()
        return solutions
    
    
    #Remove the column from the Fox-Jacobian corresponding to the given generator
    def reduce_fox_jacobi(self, gen:str = ""):
        gens = self.generators()
        rels = self.relators()
        remove = gens.index(gen)
        foxjacobi = self.fox_jacobi()
        F = self.free_algebra()
        M = MatrixSpace(F, len(rels), len(gens)-1)
        foxjacobired = M.matrix()
        for i in range(len(rels)):
            j = 0
            k = 0
            while j <= len(gens) - 1:
                if j == remove:
                    j += 1
                foxjacobired[i,k]=foxjacobi[i,j]
                j += 1
                k += 1
                if j == remove:
                    j += 1
        return foxjacobired
    
    def tap_degree(self):
        tap = self.tap()
        exp = tap.exponents()
        degree = max(exp)-min(exp)
        return degree
    
    
    #This will find an isomorphism between the triangulation and wirtinger presentations of the fundamental group.
    def triangulation_to_wirtinger(self):
        triang = magma(self.fundamental_group().magma_string())
        prewirt = self.manifold().link().knot_group()
        wirtinger = magma("Group" + str(prewirt)[25:])
        isiso = magma(False)
        i = 5
        while isiso == magma(False):
            try:
                isom = magma.SearchForIsomorphism(triang, wirtinger, i, nvals = 3)
                isiso = isom[0]
            except:
                i+=1
            return isom[1]


    


In [2]:
tapob = TAP('% Triangulation\nK[2,4]\nnot_attempted 0.0\nunknown_orientability\nCS_unknown\n0 0\n6\n   3    1    1    2 \n 0132 1023 0132 0132\n  -1   -1   -1   -1 \n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n0.0 0.0\n   0    2    3    0 \n 1023 0132 0132 0132\n  -1   -1   -1   -1 \n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n0.0 0.0\n   5    1    0    4 \n 0132 0132 0132 0132\n  -1   -1   -1   -1 \n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n0.0 0.0\n   0    4    5    1 \n 0132 0132 0132 0132\n  -1   -1   -1   -1 \n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n0.0 0.0\n   5    3    2    5 \n 2103 0132 0132 0321\n  -1   -1   -1   -1 \n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n0.0 0.0\n   2    4    4    3 \n 0132 0321 2103 0132\n  -1   -1   -1   -1 \n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0\n0.0 0.0\n'
)

/opt/SageMath/local/lib/python3.9/site-packages/sage/symbolic/expression_conversions.py:972: DeprecationWarning: invalid escape sequence \|
  """


In [71]:
manifold = 'b++RLRLRL'
print(manifold)
for i in range(len(snappy.Manifold(manifold).ptolemy_obstruction_classes())):
    tapob = TAP(manifold, obstruction = i)
    try:
        tapob.set_tap(0)
        print('Obstruction Class %d' % i)
        print(tapob.tap())
    except:
        print('No solutions with obstruction class %d' % i)
        

b++RLRLRL
No solutions with obstruction class 0
No solutions with obstruction class 1
No solutions with obstruction class 2
No solutions with obstruction class 3
No solutions with obstruction class 4
No solutions with obstruction class 5
No solutions with obstruction class 6
No solutions with obstruction class 7


In [79]:
tapob = TAP('b++RLRLRL', obstruction = 4)

In [80]:
tapob.ptolemy_solutions()

[PtolemyCoordinates({'c_1100_0': -1, 'c_0011_0': 1, 'c_1010_5': -1, 'c_0011_1': 1, 'c_1100_1': -1, 'c_0101_2': 1, 'c_1010_3': Mod(4*x^7 + 10*x^5 + 11*x^3 + 4*x, x^8 + 5/2*x^6 + 11/4*x^4 + 5/4*x^2 + 1/4), 'c_1100_4': Mod(4*x^7 + 10*x^5 + 11*x^3 + 4*x, x^8 + 5/2*x^6 + 11/4*x^4 + 5/4*x^2 + 1/4), 'c_0011_4': Mod(-4*x^7 - 10*x^5 - 11*x^3 - 4*x, x^8 + 5/2*x^6 + 11/4*x^4 + 5/4*x^2 + 1/4), 'c_0101_0': Mod(-4*x^7 - 10*x^5 - 11*x^3 - 4*x, x^8 + 5/2*x^6 + 11/4*x^4 + 5/4*x^2 + 1/4), 'c_1100_5': Mod(4*x^7 + 10*x^5 + 11*x^3 + 4*x, x^8 + 5/2*x^6 + 11/4*x^4 + 5/4*x^2 + 1/4), 'c_0011_5': Mod(-4*x^7 - 10*x^5 - 11*x^3 - 4*x, x^8 + 5/2*x^6 + 11/4*x^4 + 5/4*x^2 + 1/4), 'c_1010_4': Mod(x, x^8 + 5/2*x^6 + 11/4*x^4 + 5/4*x^2 + 1/4), 'c_0110_0': Mod(x, x^8 + 5/2*x^6 + 11/4*x^4 + 5/4*x^2 + 1/4), 'c_0110_5': Mod(x, x^8 + 5/2*x^6 + 11/4*x^4 + 5/4*x^2 + 1/4), 'c_1001_0': Mod(x, x^8 + 5/2*x^6 + 11/4*x^4 + 5/4*x^2 + 1/4), 'c_0101_1': Mod(x, x^8 + 5/2*x^6 + 11/4*x^4 + 5/4*x^2 + 1/4), 'c_1001_5': Mod(x, x^8 + 5/2*x^6 

In [81]:
tapob.set_tap(0)

ValueError: matrix equation has no solutions

In [36]:
tapob.tap()

t0^2 + 4*t0 + 1

In [90]:
from termcolor import colored

class bcolors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'
    HIGHLIGHT = '\033[100m'
    HIGHLIGHTYELLOW = '\033[103m'
    HIGHLIGHTRED = '\033[101m'
    
    
mfld_str='b++RL'
mflds_arr=[snappy.Manifold('b++RL')]
           
for i in range(10):
    mfld_str = mfld_str + 'L'
    mflds_arr.append(snappy.Manifold(mfld_str))

for M in mflds_arr:
    if M.volume() != 0:
#         tf = False
#         tapob = TAP(M, 2, "PSL")
#         tapob.set_ptolemy_solutions(obstr = 'all')
#         for i in range(len(tapob.ptolemy_solutions())):
#             tapob.set_ptolemy_solutions(obstr = i)
#             for j in range(len(tapob.ptolemy_solutions())):    
#                 tap = tapob.make_TAP(j)
#                 if tap.is_monic():
#                     tf = True
#         if tf:
            tapob = TAP(M, 2, "PSL")
            tapob.set_ptolemy_solutions(obstr = 'all')
            print("The following are the TAP associated to the parabolic representations of {}".format(M))
            print(tapob.relators())
            for i in range(len(tapob.ptolemy_solutions())):
                if i == 0:
                    print('Obstruction Class: ', colored('Trivial', 'red'))
                else:
                    print('Obstruction Class: ', colored('Non-Trivial', 'red'))
                tapob.set_ptolemy_solutions(obstr = i)   
                for j in range(len(tapob.ptolemy_solutions())):
                    tapob.set_tap(j)
                    tap = tapob.tap()
                    print(bcolors.HIGHLIGHT, tap, bcolors.ENDC)
                    print(tapob.number_field())
                    print(bcolors.HIGHLIGHTRED, tapob.tap_degree(), bcolors.ENDC)
                    leading_coef = tap.coefficients()[0]
                    if leading_coef != 1:
                        print(bcolors.HIGHLIGHTYELLOW, "False", bcolors.ENDC)
                    print("")
                print("")
            print("")
            print("")

The following are the TAP associated to the parabolic representations of b++RL(0,0)
['aaBAbbbAB']
Obstruction Class:  Trivial

Obstruction Class:  Non-Trivial
 t0^2 + 4*t0 + 1 
Number Field in x with defining polynomial x^2 + x + 1
 2 




The following are the TAP associated to the parabolic representations of b++RLL(0,0)
['aaBAbaabAB']
Obstruction Class:  Trivial

Obstruction Class:  Non-Trivial

Obstruction Class:  Non-Trivial
 t0^2 + (4*x^3 - 4)*t0 + 1 
Number Field in x with defining polynomial x^4 + 1/2*x^2 + 1/2
 2 


Obstruction Class:  Non-Trivial



The following are the TAP associated to the parabolic representations of b++RLLL(0,0)
['aabABaaaBAb']
Obstruction Class:  Trivial

Obstruction Class:  Non-Trivial
 t0^2 + (8*x^3 + 4*x^2 + 4*x + 8)*t0 + 1 
Number Field in x with defining polynomial x^4 + 1/2*x^3 + x^2 + 1/2*x + 1/2
 2 




The following are the TAP associated to the parabolic representations of b++RLLLL(0,0)
['aaaBAbaaabAB']
Obstruction Class:  Trivial

Obstruction

In [105]:
from termcolor import colored

class bcolors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'
    HIGHLIGHT = '\033[100m'
    HIGHLIGHTYELLOW = '\033[103m'
    HIGHLIGHTRED = '\033[101m'
    
    
mflds_arr=[]
           
for i in range(10):
    mflds_arr.append(snappy.RationalTangle(2*i+3, 2).numerator_closure().exterior())

for M in mflds_arr:
    if M.volume() != 0:
#         tf = False
#         tapob = TAP(M, 2, "PSL")
#         tapob.set_ptolemy_solutions(obstr = 'all')
#         for i in range(len(tapob.ptolemy_solutions())):
#             tapob.set_ptolemy_solutions(obstr = i)
#             for j in range(len(tapob.ptolemy_solutions())):    
#                 tap = tapob.make_TAP(j)
#                 if tap.is_monic():
#                     tf = True
#         if tf:
            tapob = TAP(M, 2, "PSL")
            tapob.set_ptolemy_solutions(obstr = 'all')
            print("The following are the TAP associated to the parabolic representations of {}".format(snappy.LinkExteriors.identify(M)))
            print(tapob.relators())
            for i in range(len(tapob.ptolemy_solutions())):
                if i == 0:
                    print('Obstruction Class: ', colored('Trivial', 'red'))
                else:
                    print('Obstruction Class: ', colored('Non-Trivial', 'red'))
                tapob.set_ptolemy_solutions(obstr = i)   
                for j in range(len(tapob.ptolemy_solutions())):
                    tapob.set_tap(j)
                    tap = tapob.tap()
                    print(bcolors.HIGHLIGHT, tap, bcolors.ENDC)
                    print(tapob.number_field())
                    print(bcolors.HIGHLIGHTRED, tapob.tap_degree(), bcolors.ENDC)
                    leading_coef = tap.coefficients()[0]
                    if leading_coef != 1:
                        print(bcolors.HIGHLIGHTYELLOW, "False", bcolors.ENDC)
                    print("")
                print("")
            print("")
            print("")

The following are the TAP associated to the parabolic representations of 4_1(0,0)
['aaBAbbbAB']
Obstruction Class:  Trivial

Obstruction Class:  Non-Trivial
 t0^2 - 4*t0 + 1 
Number Field in x with defining polynomial x^2 + x + 1
 2 




The following are the TAP associated to the parabolic representations of 5_2(0,0)
['abbaBAABabbaB']
Obstruction Class:  Trivial

Obstruction Class:  Non-Trivial
 (x^2 + 2*x + 5)*t0^2 - 4*t0 + x^2 + 2*x + 5 
Number Field in x with defining polynomial x^3 + 2*x^2 + 3*x + 1
 2 
 False 




The following are the TAP associated to the parabolic representations of 6_1(0,0)
['aBAbbbABabABaaBAb']
Obstruction Class:  Trivial

Obstruction Class:  Non-Trivial
 (x^3 + 2*x^2 + x + 3)*t0^2 + (-4*x^3 - 8*x^2 - 4*x - 8)*t0 + x^3 + 2*x^2 + x + 3 
Number Field in x with defining polynomial x^4 + 3*x^3 + 4*x^2 + 3*x + 2
 2 
 False 




The following are the TAP associated to the parabolic representations of 7_2(0,0)
['aaabABabABabbaBAbaBAb']
Obstruction Class:  Trivial



In [10]:
tapob.fox_jacobi_to_twisted_alexander_matrix('b').det()

(-x^3 - 4*x) + (-4*x^3 - 12*x)*t0 + (-x^3 - 4*x)*t0^2

In [15]:
(identity_matrix(tapob.dimension()) - tapob.lift_rep()[tapob.generators().index('a')]*tapob.make_rep('a')).det()

1 + (2*x + 2)*t0 + t0^2

In [16]:
identity_matrix(tapob.dimension()) - tapob.lift_rep()[tapob.generators().index('a')]*tapob.make_rep('a')

[     1 + (x^2 + 4*x + 3)*t0                  (x + 2)*t0]
[(-x^3 - 4*x^2 - 4*x - 2)*t0     1 + (-x^2 - 2*x - 1)*t0]

In [19]:
tapob.fundamental_group()

Generators:
   a,b
Relators:
   aaBAbaBAbbbABabAB

In [20]:
TAP('7_7').ptolemy_solutions().evaluate_word('a', G)[1]

[[Mod(-2*x^3 - 6*x^2 - 4*x + 1, x^4 + 3*x^3 + x^2 - 2*x + 1),
  Mod(-2*x^3 - 8*x^2 - 8*x, x^4 + 3*x^3 + x^2 - 2*x + 1)],
 [Mod(2*x^3 + 4*x^2 + 1, x^4 + 3*x^3 + x^2 - 2*x + 1),
  Mod(2*x^2 + 4*x + 1, x^4 + 3*x^3 + x^2 - 2*x + 1)]]

In [49]:
TAP('m003').ptolemy_solutions()

[PtolemyCoordinates({'c_1100_0': 1, 'c_1001_0': -1, 'c_0011_0': 1, 'c_0011_1': -1, 'c_1001_1': 1, 'c_1100_1': -1, 'c_0101_0': Mod(x, x^2 + x + 1), 'c_0110_1': Mod(x, x^2 + x + 1), 'c_0110_0': Mod(-x, x^2 + x + 1), 'c_0101_1': Mod(-x, x^2 + x + 1), 'c_1010_0': Mod(x, x^2 + x + 1), 'c_1010_1': Mod(-x, x^2 + x + 1), 's_3_0': 1, 's_2_0': -1, 's_1_0': 1, 's_0_0': -1, 's_0_1': -1, 's_1_1': 1, 's_2_1': -1, 's_3_1': 1}, is_numerical = False, ...)]

In [50]:
tap = TAP('m003')

In [52]:
tap.set_tap(0)

In [36]:
G = tap.fundamental_group()

In [8]:
tap.fox_jacobi()[0,0]

Id + Id*a - Id*a^2*c*A - Id*a^2*c*A^2 - Id*a^2*c*A^2*B*A + Id*a^2*c*A^2*B*A*c + Id*a^2*c*A^2*B*A*c*a

In [37]:
tap.set_solution(0)

In [38]:
tap.init_number_field()

In [39]:
tap.init_polynomial_ring()

In [40]:
tap.lift_rep()

[t0, 1]

In [199]:
tap.ptolemy_solutions().evaluate_word('a', G)

[[[Mod(-x^2 + 2, x^5 - x^4 - 3*x^3 + 2*x^2 + 2*x + 1), Mod(-x^4 + x^3 + 3*x^2 - 2*x - 2, x^5 - x^4 - 3*x^3 + 2*x^2 + 2*x + 1)], [Mod(-x, x^5 - x^4 - 3*x^3 + 2*x^2 + 2*x + 1), Mod(0, x^5 - x^4 - 3*x^3 + 2*x^2 + 2*x + 1)]], [[Mod(-x^2 + 2, x^6 - x^5 - 2*x^4 + 2*x^2 + 2*x - 1), Mod(x^5 - x^4 - 2*x^3 + 2*x + 2, x^6 - x^5 - 2*x^4 + 2*x^2 + 2*x - 1)], [Mod(-x, x^6 - x^5 - 2*x^4 + 2*x^2 + 2*x - 1), Mod(0, x^6 - x^5 - 2*x^4 + 2*x^2 + 2*x - 1)]]]

In [203]:
matA = t^2 * tap.make_rep('a')

In [202]:
matB = t^-4 * tap.make_rep('b')

In [201]:
matC = t^-1 * tap.make_rep('c')

In [244]:
matrix(tap.polynomial_ring(), ((tap.fox_jacobi()[0,0](*tup)[0,0],tap.fox_jacobi()[0,0](*tup)[0,1], tap.fox_jacobi()[0,1](*tup)[0,0], tap.fox_jacobi()[0,1](*tup)[0,1]),(tap.fox_jacobi()[0,0](*tup)[1,0],tap.fox_jacobi()[0,0](*tup)[1,1], tap.fox_jacobi()[0,1](*tup)[1,0], tap.fox_jacobi()[0,1](*tup)[1,1]),(tap.fox_jacobi()[1,0](*tup)[0,0],tap.fox_jacobi()[1,0](*tup)[0,1], tap.fox_jacobi()[1,1](*tup)[0,0], tap.fox_jacobi()[1,1](*tup)[0,1]),(tap.fox_jacobi()[1,0](*tup)[1,0],tap.fox_jacobi()[1,0](*tup)[1,1], tap.fox_jacobi()[1,1](*tup)[1,0], tap.fox_jacobi()[1,1](*tup)[1,1])))[0,2]#/(matrix.identity(2)-matC).det()

(x^4 + x^3 - 3*x^2 + x + 2)*t^3 + (x^4 + x^3 - 4*x^2 - x + 3)*t^4

In [207]:
tup = (matrix.identity(2), matA, matA.inverse(), matB, matB.inverse(), matC, matC.inverse())

In [41]:
tap.fox_jacobi_to_twisted_alexander_matrix('a')

[-1 + (-4*x + 5)*t0 + (4*x - 4)*t0^2      (-2*x - 1)*t0 + (2*x - 3)*t0^2]
[       (x - 3)*t0 + (-2*x + 3)*t0^2   -1 + (2*x + 1)*t0 + (-x + 2)*t0^2]

In [183]:
tap.free_algebra()

Free Algebra on 5 generators (Id, a, A, b, B) over Integer Ring

In [187]:
((tap.fox_jacobi()[0,0](matrix.identity(2), t*matA, t^-1*matA.inverse(), matB, matB.inverse()).det())/(matrix.identity(2)-matB).det()).numerator().numerator()

11*t^12 - 88*t^10 + (30*x^5 - 108*x^4 + 259*x^3 - 443*x^2 + 365*x + 200)*t^8 + (-48*x^5 + 164*x^4 - 388*x^3 + 656*x^2 - 496*x - 320)*t^6 + (30*x^5 - 108*x^4 + 259*x^3 - 443*x^2 + 365*x + 200)*t^4 - 88*t^2 + 11

In [177]:
tap.preabelianize()

(1, 0)

In [185]:
tap.number_field()

Number Field in x with defining polynomial x^6 - 4*x^5 + 10*x^4 - 18*x^3 + 18*x^2 - 7*x + 1

In [212]:
tap.fox_jacobi()

[Id + Id*a - Id*a^2*c*A - Id*a^2*c*A^2 - Id*a^2*c*A^2*B*A + Id*a^2*c*A^2*B*A*c + Id*a^2*c*A^2*B*A*c*a                                                             -Id*a^2*c*A^2*B + Id*a^2*c*A^2*B*A*c*a^2                                                                            Id*a^2 + Id*a^2*c*A^2*B*A]
[                                                                                     Id + Id*a*b*C*B                                                                   Id*a - Id*a*b*C*B + Id*a*b*C*B*a*c                                                                             -Id*a*b*C + Id*a*b*C*B*a]

In [215]:
tap.reduce_fox_jacobi('a')

[-Id*a^2*c*A^2*B + Id*a^2*c*A^2*B*A*c*a^2                Id*a^2 + Id*a^2*c*A^2*B*A]
[      Id*a - Id*a*b*C*B + Id*a*b*C*B*a*c                 -Id*a*b*C + Id*a*b*C*B*a]

In [53]:
tap.tap()

t0^2 + (-4*x - 2)*t0 + 1

In [44]:
TAP('(a)^2*B', obstruction = 1, twister = True).manifold().ptolemy_variety(2,'all')

[Ptolemy Variety for (a)^2*B, N = 2, obstruction_class = 0
    - c_0011_0 * c_0011_2 - c_0011_0^2 + c_0101_0^2
    - c_0011_0^2 + c_0011_2^2 - c_0101_0^2
    c_0011_0 * c_0011_2 - c_0011_0^2 + c_0101_0^2
    - 1 + c_0011_0, Ptolemy Variety for (a)^2*B, N = 2, obstruction_class = 1
    - c_0011_0 * c_0011_2 + c_0011_0^2 - c_0101_0^2
    c_0011_0^2 - c_0011_2^2 - c_0101_0^2
    c_0011_0 * c_0011_2 - c_0011_0^2 + c_0101_0^2
    - 1 + c_0011_0, Ptolemy Variety for (a)^2*B, N = 2, obstruction_class = 2
    c_0011_0 * c_0011_2 + c_0011_0^2 + c_0101_0^2
    c_0011_0^2 + c_0011_2^2 + c_0101_0^2
    - c_0011_0 * c_0011_2 + c_0011_0^2 + c_0101_0^2
    - 1 + c_0011_0, Ptolemy Variety for (a)^2*B, N = 2, obstruction_class = 3
    c_0011_0 * c_0011_2 - c_0011_0^2 - c_0101_0^2
    - c_0011_0^2 - c_0011_2^2 + c_0101_0^2
    - c_0011_0 * c_0011_2 + c_0011_0^2 + c_0101_0^2
    - 1 + c_0011_0]

In [42]:
TAP('(a)^2*B', obstruction = 3, twister = True).ptolemy_solutions()

[PtolemyCoordinates({'c_1001_0': 1, 'c_0110_0': -1, 'c_1100_2': 1, 'c_0011_1': -1, 'c_1100_1': 1, 'c_0011_0': 1, 'c_1001_2': 1, 'c_0110_2': -1, 'c_0101_1': Mod(x, x^4 + x^2 + 2), 'c_1010_0': Mod(-x, x^4 + x^2 + 2), 'c_1010_2': Mod(-x, x^4 + x^2 + 2), 'c_0101_0': Mod(x, x^4 + x^2 + 2), 'c_0101_2': Mod(x, x^4 + x^2 + 2), 'c_1010_1': Mod(x, x^4 + x^2 + 2), 'c_1100_0': Mod(-x^2 - 1, x^4 + x^2 + 2), 'c_1001_1': Mod(x^2 + 1, x^4 + x^2 + 2), 'c_0110_1': Mod(x^2 + 1, x^4 + x^2 + 2), 'c_0011_2': Mod(x^2 + 1, x^4 + x^2 + 2), 's_2_1': -1, 's_0_1': 1, 's_3_0': -1, 's_2_0': 1, 's_1_0': 1, 's_0_0': -1, 's_2_2': -1, 's_3_2': 1, 's_1_1': 1, 's_3_1': -1, 's_1_2': 1, 's_0_2': -1}, is_numerical = False, ...)]

t^6 - 4*t^5 + (x^2 - 2*x + 4)*t^4 + (8*x^4 - 20*x^3 + 36*x + 12)*t^3 + (x^2 - 2*x + 4)*t^2 - 4*t + 1

In [12]:
from termcolor import colored

class bcolors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'
    HIGHLIGHT = '\033[100m'
    HIGHLIGHTYELLOW = '\033[103m'
    HIGHLIGHTRED = '\033[101m'

for M in snappy.LinkExteriors[0:100]:
    if M.volume() != 0:
#         tf = False
#         tapob = TAP(M, 2, "PSL")
#         tapob.set_ptolemy_solutions(obstr = 'all')
#         for i in range(len(tapob.ptolemy_solutions())):
#             tapob.set_ptolemy_solutions(obstr = i)
#             for j in range(len(tapob.ptolemy_solutions())):    
#                 tap = tapob.make_TAP(j)
#                 if tap.is_monic():
#                     tf = True
#         if tf:
            tapob = TAP(M, 2, "PSL")
            tapob.set_ptolemy_solutions(obstr = 'all')
            print("The following are the TAP associated to the parabolic representations of {}".format(M))
            print(tapob.relators())
            for i in range(len(tapob.ptolemy_solutions())):
                if i == 0:
                    print('Obstruction Class: ', colored('Trivial', 'red'))
                else:
                    print('Obstruction Class: ', colored('Non-Trivial', 'red'))
                tapob.set_ptolemy_solutions(obstr = i)   
                for j in range(len(tapob.ptolemy_solutions())):
                    tapob.set_tap(j)
                    tap = tapob.tap()
                    print(bcolors.HIGHLIGHT, tap, bcolors.ENDC)
                    print(tapob.number_field())
                    print(bcolors.HIGHLIGHTRED, tapob.tap_degree(), bcolors.ENDC)
                    leading_coef = tap.coefficients()[0]
                    if leading_coef != 1:
                        print(bcolors.HIGHLIGHTYELLOW, "False", bcolors.ENDC)
                    print("")
                print("")
            print("")
            print("")

The following are the TAP associated to the parabolic representations of4_1(0,0)
['abbbaBAAB']
Obstruction Class:  Trivial

Obstruction Class:  Non-Trivial
 t^2 - 4*t + 1 
Number Field in x with defining polynomial x^2 - x + 1
 2 




The following are the TAP associated to the parabolic representations of5_2(0,0)
['aBBBabbAAbb']
Obstruction Class:  Trivial

Obstruction Class:  Non-Trivial
 (x^2 + 2*x + 5)*t^2 - 4*t + x^2 + 2*x + 5 
Number Field in x with defining polynomial x^3 + 2*x^2 + 3*x + 1
 2 
 False 




The following are the TAP associated to the parabolic representations of6_1(0,0)
['aBAbaBAABabABabbb']
Obstruction Class:  Trivial

Obstruction Class:  Non-Trivial
 (x^3 - 2*x^2 + x + 5)*t^2 + (4*x^3 - 8*x^2 + 4*x + 16)*t + x^3 - 2*x^2 + x + 5 
Number Field in x with defining polynomial x^4 - 2*x^3 + x^2 + 3*x + 1
 2 
 False 




The following are the TAP associated to the parabolic representations of6_2(0,0)
['aaBBBaaBAbbbAAbbbAB']
Obstruction Class:  Trivial

Obstruction Clas

In [6]:
from termcolor import colored

class bcolors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'
    HIGHLIGHT = '\033[100m'
    HIGHLIGHTYELLOW = '\033[103m'
    HIGHLIGHTRED = '\033[101m'

for M in snappy.OrientableCuspedCensus[0:100]:
    if M.volume() != 0:
#         tf = False
#         tapob = TAP(M, 2, "PSL")
#         tapob.set_ptolemy_solutions(obstr = 'all')
#         for i in range(len(tapob.ptolemy_solutions())):
#             tapob.set_ptolemy_solutions(obstr = i)
#             for j in range(len(tapob.ptolemy_solutions())):    
#                 tap = tapob.make_TAP(j)
#                 if tap.is_monic():
#                     tf = True
#         if tf:
            tapob = TAP(M, 2, "PSL")
            tapob.set_ptolemy_solutions(obstr = 'all')
            if snappy.LinkExteriors.identify(M):
                print("The following are the TAP associated to the parabolic representations of {}, the knot exterior of {}".format(M, snappy.LinkExteriors.identify(M)))
            else:
                print("The following are the TAP associated to the parabolic representations of {}".format(M))
            print(tapob.relators())
            for i in range(len(tapob.ptolemy_solutions())):
                if i == 0:
                    print('Obstruction Class: ', colored('Trivial', 'red'))
                else:
                    print('Obstruction Class: ', colored('Non-Trivial', 'red'))
                    
                try:
                    tapob.set_ptolemy_solutions(obstr = i)   
                    for j in range(len(tapob.ptolemy_solutions())):
                        tapob.set_tap(j)
                        tap = tapob.tap()
                        print(bcolors.HIGHLIGHT, tap, bcolors.ENDC)
                        print(tapob.number_field())
                        print(bcolors.HIGHLIGHTRED, tapob.tap_degree(), bcolors.ENDC)
                        leading_coef = tap.coefficients()[0]
                        if leading_coef != 1:
                            print(bcolors.HIGHLIGHTYELLOW, "False", bcolors.ENDC)
                        print("")
                except:
                    print(bcolors.OKBLUE, bcolors.BOLD, bcolors.HIGHLIGHTYELLOW, "PSL(2,C) representation does not lift to SL(2,C)", bcolors.ENDC)
                print("")
            print("")
            print("")

The following are the TAP associated to the parabolic representations of m003(0,0)
['abAAbabbb']
Obstruction Class:  Trivial
 t0^2 - 2*t0 + 1 
Number Field in x with defining polynomial x^2 - x - 1
 2 


Obstruction Class:  Non-Trivial
 t0^2 + (-4*x - 2)*t0 + 1 
Number Field in x with defining polynomial x^2 + x + 1
 2 




The following are the TAP associated to the parabolic representations of m004(0,0), the knot exterior of 4_1(0,0)
['aaabABBAb']
Obstruction Class:  Trivial

Obstruction Class:  Non-Trivial
 t0^2 - 4*t0 + 1 
Number Field in x with defining polynomial x^2 + x + 1
 2 




The following are the TAP associated to the parabolic representations of m006(0,0)
['ababbAAbb']
Obstruction Class:  Trivial
 (-x + 1)*t0^2 + (2*x - 2)*t0 - x + 1 
Number Field in x with defining polynomial x^2 + x - 1
 2 
 False 


Obstruction Class:  Non-Trivial
 (x^2 + 4)*t0^2 + (4*x^2 + 2*x + 6)*t0 + x^2 + 4 
Number Field in x with defining polynomial x^3 + 2*x - 1
 2 
 False 




The following ar

In [3]:
from termcolor import colored

class bcolors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'
    HIGHLIGHT = '\033[100m'
    HIGHLIGHTYELLOW = '\033[103m'
    HIGHLIGHTRED = '\033[101m'

for M in snappy.LinkExteriors[0:100]:
    if M.volume() != 0:
#         tf = False
#         tapob = TAP(M, 2, "PSL")
#         tapob.set_ptolemy_solutions(obstr = 'all')
#         for i in range(len(tapob.ptolemy_solutions())):
#             tapob.set_ptolemy_solutions(obstr = i)
#             for j in range(len(tapob.ptolemy_solutions())):    
#                 tap = tapob.make_TAP(j)
#                 if tap.is_monic():
#                     tf = True
#         if tf:
            tapob = TAP(M, 2, "PSL")
            tapob.set_ptolemy_solutions(obstr = 'all')
            print("The following are the TAP associated to the parabolic representations of {}".format(M))
            print(tapob.relators())
            for i in range(len(tapob.ptolemy_solutions())):
                if i == 0:
                    print('Obstruction Class: ', colored('Trivial', 'red'))
                else:
                    print('Obstruction Class: ', colored('Non-Trivial', 'red'))
                tapob.set_ptolemy_solutions(obstr = i)   
                for j in range(len(tapob.ptolemy_solutions())):
                    tapob.set_tap(j)
                    tap = tapob.tap()
                    print(bcolors.HIGHLIGHT, tap, bcolors.ENDC)
                    print(tapob.number_field())
                    print(bcolors.HIGHLIGHTRED, tapob.tap_degree(), bcolors.ENDC)
                    leading_coef = tap.coefficients()[0]
                    if leading_coef != 1:
                        print(bcolors.HIGHLIGHTYELLOW, "False", bcolors.ENDC)
                    print("")
                print("")
            print("")
            print("")

/opt/SageMath/local/lib/python3.9/site-packages/sage/symbolic/expression_conversions.py:972: DeprecationWarning: invalid escape sequence \|
  """


The following are the TAP associated to the parabolic representations of 4_1(0,0)
['abbbaBAAB']
Obstruction Class:  Trivial

Obstruction Class:  Non-Trivial
 t0^2 - 4*t0 + 1 
Number Field in x with defining polynomial x^2 - x + 1
 2 




The following are the TAP associated to the parabolic representations of 5_2(0,0)
['aBBBabbAAbb']
Obstruction Class:  Trivial

Obstruction Class:  Non-Trivial
 (x^2 + 2*x + 5)*t0^2 - 4*t0 + x^2 + 2*x + 5 
Number Field in x with defining polynomial x^3 + 2*x^2 + 3*x + 1
 2 
 False 




The following are the TAP associated to the parabolic representations of 6_1(0,0)
['aBAbaBAABabABabbb']
Obstruction Class:  Trivial

Obstruction Class:  Non-Trivial
 (x^3 - 2*x^2 + x + 5)*t0^2 + (4*x^3 - 8*x^2 + 4*x + 16)*t0 + x^3 - 2*x^2 + x + 5 
Number Field in x with defining polynomial x^4 - 2*x^3 + x^2 + 3*x + 1
 2 
 False 




The following are the TAP associated to the parabolic representations of 6_2(0,0)
['aaBBBaaBAbbbAAbbbAB']
Obstruction Class:  Trivial

Obstru

KeyboardInterrupt: 